In [1]:
# Load only from python 3.12
import sys

sys.path = [p for p in sys.path if "python3.8" not in p]

In [18]:
#!{sys.executable} -m pip install aiohttp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 5.0 MB/s  0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [aiohttp]m6/7 [aiohttp]l]eballs]


In [20]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import rasterio
import numpy as np

In [4]:
# Load 1x1km grid
grid_1km = pd.read_parquet("data/grid_1km.parquet")

In [5]:
# Define coordinate offset from lower-left corner
offsets = [
    (0, 0),     # Bottom-Left
    (500, 0),   # Bottom-Right
    (0, 500),   # Top-Left
    (500, 500)  # Top-Right
]

In [7]:
# Create new grid
new_grids = []

# Iterate through offset
for dx, dy in offsets:
    df_temp = grid_1km.copy()
    
    # Calculate new coordinates based on GISCO column names
    df_temp['X_LLC_500'] = df_temp['X_LLC'] + dx
    df_temp['Y_LLC_500'] = df_temp['Y_LLC'] + dy
    
    new_grids.append(df_temp)

In [8]:
# Combine into 500m grid df
df_500m = pd.concat(new_grids, ignore_index=True)

In [9]:
# Create geometries
geometries = [
    box(x, y, x + 500, y + 500) 
    for x, y in zip(df_500m['X_LLC_500'], df_500m['Y_LLC_500'])
]

In [10]:
# Create gdf
grid_500m = gpd.GeoDataFrame(df_500m, geometry=geometries, crs="EPSG:3035")

In [11]:
# Save as geoparquet
grid_500m.to_parquet('data/grid_500m.parquet')

## Filter with GHSL raster

In [13]:
# Define raster
ghsl_path = "data/GHSL_europe.tif"

target_values = [
    13, # Rural cluster
    21, # Suburban or peri-urban
    22, # Semi-dense urban cluster
    23, # Dense urban cluster
    30 # Urban centre
]

In [16]:
# Raster value sample
with rasterio.open(ghsl_path) as src:
    raster = src.read(1)
    transform = src.transform

    # Get centroid of each grid cell
    coords = [(geom.centroid.x, geom.centroid.y) for geom in grid_500m.geometry]

    # Sample raster at centroids
    sampled_vals = list(src.sample(coords))
    sampled_vals = np.array(sampled_vals).flatten()

In [17]:
# Filter
mask = np.isin(sampled_vals, target_values)
filtered_grid = grid_500m[mask]

In [27]:
# Rename grid id
filtered_grid["GRD_ID"] = (
    "N" + filtered_grid["Y_LLC_500"].astype(str) +
    "E" + filtered_grid["X_LLC_500"].astype(str)
)

In [15]:
# Read only relevant column
filtered_grid = filtered_grid[["GRD_ID", "CNTR_ID", "X_LLC_500", "Y_LLC_500", "geometry"]]

In [17]:
# Save filter
filtered_grid.to_parquet('data/grid_500m_filtered.parquet')

## Image acquisition

In [14]:
# Read grid
filtered_grid = gpd.read_parquet('data/grid_500m_filtered.parquet')

In [11]:
nl_grid = filtered_grid[filtered_grid["CNTR_ID"].str.contains("NL")] 

In [16]:
filtered_grid

,GRD_ID,CNTR_ID,X_LLC_500,Y_LLC_500,geometry
3,N-3028000E9988000,FR,9988000,-3028000,"POLYGON ((9988500 -3028000, 9988500 -3027500, ..."
19,N2089000E6024000,TR,6024000,2089000,"POLYGON ((6024500 2089000, 6024500 2089500, 60..."
24,N2089000E6029000,TR,6029000,2089000,"POLYGON ((6029500 2089000, 6029500 2089500, 60..."
55,N2089000E6057000,TR,6057000,2089000,"POLYGON ((6057500 2089000, 6057500 2089500, 60..."
56,N2089000E6058000,TR,6058000,2089000,"POLYGON ((6058500 2089000, 6058500 2089500, 60..."
...,...,...,...,...,...
28220753,N2089500E5875500,TR,5875500,2089500,"POLYGON ((5876000 2089500, 5876000 2090000, 58..."
28220810,N2089500E5926500,TR,5926500,2089500,"POLYGON ((5927000 2089500, 5927000 2090000, 59..."
28220813,N2089500E5929500,TR,5929500,2089500,"POLYGON ((5930000 2089500, 5930000 2090000, 59..."
28220815,N2089500E5931500,TR,5931500,2089500,"POLYGON ((5932000 2089500, 5932000 2090000, 59..."
